# D2：动作执行以前，先检查后果

直接 VLA 给出动作以后就结束了。这里训练一个独立 outcome model，预测下一状态和碰撞，再对候选动作重排。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.robot import TabletopOutcomeModel, make_outcome_dataset, outcome_loss, rerank_actions, step_tabletop
torch.manual_seed(1)


## 1. 动作后果数据与 VLA 示范不同

若只收专家安全动作，碰撞 head 几乎看不到失败。Outcome 数据故意加入随机候选动作与碰撞。

In [ ]:
data = make_outcome_dataset(400, seed=1)
print('collision ratio:', round(float(data['collisions'].mean()), 3))
assert 0 < data['collisions'].mean() < 1


## 2. 学习 `state + action → next state + collision`

这个模型没有语言。语言负责指定目标；世界模型只学习动作怎样改变桌面状态。

In [ ]:
model = TabletopOutcomeModel()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
losses = []
for _ in range(80):
    opt.zero_grad(); loss, state_loss, collision_loss = outcome_loss(model, data['states'], data['actions'], data['next_states'], data['collisions']); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('outcome loss:', round(losses[0], 3), '→', round(losses[-1], 3))
print('state/collision:', round(float(state_loss), 3), round(float(collision_loss), 3))
assert losses[-1] < losses[0]


## 3. 候选动作重排

我们构造一个障碍挡在抓手与红色目标之间的状态。直接向右很短，但会碰撞；斜向动作稍远，却更安全。

In [ ]:
state = torch.tensor([0.20, 0.50, 0.85, 0.50, 0.20, 0.85, 0.31, 0.50])
candidates = torch.tensor([[1.0, 0.0], [0.7, -0.7], [0.7, 0.7], [-1.0, 0.0]])
chosen, scores = rerank_actions(model, state, instruction=0, candidates=candidates)
print('scores:', [round(float(x), 3) for x in scores], 'chosen:', chosen)
true_results = [step_tabletop(state.numpy(), action.numpy())[1] for action in candidates]
print('真实碰撞:', true_results)


## 4. 模型不一定检查正确

如果重排仍选择碰撞动作，这不是 Notebook 失败，而是一条诊断：训练覆盖、碰撞权重或 outcome model 不够。PA 必须比较真实闭环结果，不能只信模型自己的评分。

In [ ]:
with torch.no_grad():
    repeated = state[None].expand(len(candidates), -1)
    _, logits = model(repeated, candidates)
print('预测碰撞概率:', [round(float(x), 3) for x in torch.sigmoid(logits)])
print('direct candidate collision:', true_results[0], 'reranked collision:', true_results[chosen])


## 小结

VLA 提出动作，outcome model 预测后果，reranker 比较目标距离与碰撞。两个模块分开以后，我们能判断失败来自‘动作提得差’还是‘后果想错了’。